# Day 12: バッチ効果と再現性

## この章で解く現実の課題

PCAでサンプルが分かれて見えるとき、それは炎症刺激の差なのか、実験日の差なのかを見分けます。

バッチ効果とは、条件とは別の実験上の要因によって生じる系統的な差です。RNA-seqでは、ライブラリ調製日、試薬ロット、測定laneなどがバッチになり得ます。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(7)

genes = [f"gene_{i:03d}" for i in range(80)]
samples = ["C1", "C2", "C3", "T1", "T2", "T3"]
metadata = pd.DataFrame({
    "sample": samples,
    "condition": ["control", "control", "control", "treated", "treated", "treated"],
    "batch": ["batch1", "batch1", "batch1", "batch2", "batch2", "batch2"],
})

# 条件効果が小さく、バッチ効果が大きいデータを作る。
base = np.random.normal(8, 0.5, size=(len(genes), len(samples)))
condition_effect = np.array([0, 0, 0, 0.4, 0.4, 0.4])
batch_effect = np.array([0, 0, 0, 2.0, 2.0, 2.0])
expr = pd.DataFrame(base + condition_effect + batch_effect, index=genes, columns=samples)
expr.head()


In [ ]:
# PCAをNumPyだけで計算する。
# scikit-learnを使わず、Colabでもローカルvenvでも動きやすくするため。
# PCAでは「サンプル x 遺伝子」の行列にし、各遺伝子を平均0に中心化する。
X = expr.T.to_numpy()
X_centered = X - X.mean(axis=0)

# SVD: X = U S Vt。サンプル座標は U * S で得られる。
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
coords = U[:, :2] * S[:2]

pca_df = pd.DataFrame(coords, columns=["PC1", "PC2"])
pca_df = pd.concat([metadata, pca_df], axis=1)
pca_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)

for condition, group in pca_df.groupby("condition"):
    axes[0].scatter(group["PC1"], group["PC2"], label=condition, s=80)
axes[0].set_title("Color by condition")
axes[0].legend()

for batch, group in pca_df.groupby("batch"):
    axes[1].scatter(group["PC1"], group["PC2"], label=batch, s=80)
axes[1].set_title("Color by batch")
axes[1].legend()

for ax in axes:
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.tight_layout()
plt.show()


## 読み取り

この例ではconditionとbatchが完全に重なっています。そのため、PCAでcontrolとtreatedが分かれて見えても、それが刺激の効果なのかbatchの効果なのか区別できません。

重要なのは、PCAの分離そのものを喜ぶことではなく、metadataで色分けして「何によって分かれているのか」を疑うことです。


In [ ]:
pd.crosstab(metadata["batch"], metadata["condition"])


## 実務上の対策

- 実験前: conditionが各batchに均等に入るようにランダム化する。
- 解析前: metadataにbatch列を必ず残す。
- 可視化: PCAをconditionとbatchの両方で色分けする。
- 統計モデル: 可能ならdesignにbatchを含める。
- 解釈: 完全交絡している場合、統計モデルだけでは救えない。

## エラーが出た場合

- `No module named sklearn`: Colabでは通常利用できます。ローカルならvenvにscikit-learnが必要です。
- PCAが実行できない: `expr.T` のように、サンプルが行、遺伝子が列になる向きで渡してください。

## この章のゴール

PCAの群分離を見たとき、conditionだけでなくbatchでも色分けし、交絡の危険を説明できれば合格です。
